# AI Arena — TACTICAL training (suppression fire + reload-aware + cover fire)

**Goal**: train an NN that fires for *pressure* even without a clean line-of-sight kill,
hides while reloading, and lays cover fire while teammates push.

Differences from `continue_ppo_cqb`:
- New env mechanics in `combat_env.py`:
  - 30-round mag + 80-frame auto-reload
  - Suppression fire — agent can shoot in facing direction without a LoS-locked target
  - Reload-aware reward: +safe_reload while LoS-blocked, -exposed_reload while visible
  - Cover-fire reward: +reward when firing AND a teammate is closer to a visible enemy
  - Suppression credit: +reward per bullet that passes within 60u of an enemy
- Resume from `model_warrior.onnx` (aggressive baseline).
- Curriculum locked to stage 4 (deployment 1200×1200, 700u spawn).

Output: `model_tactical.onnx` — drop into `ai_arena/onnx/` for the JS NN registry.


## 1. Config

In [ ]:
# === Time budget ===
TIME_LIMIT_HOURS = 3.0       # 1 hour by default — bump if you have more time

# === Skip-toggle for smoke-test verification ===
SMOKE_TEST       = False
if SMOKE_TEST:
    TIME_LIMIT_HOURS = 0.05  # 3 minutes

# === Continued-training-specific knobs ===
# Lower entropy than the original training (0.02 → 0.005) → 0.005 → 0.001
ENT_COEF_INIT_CONT  = 0.005
ENT_COEF_FINAL_CONT = 0.001

# Light shaping for the first 10% of continued training (helps the policy
# stay 'centered' while it learns the team-1 POV), then zero
SHAPING_DECAY_FRAC  = 0.10

# === PPO hyperparams (same as original except entropy) ===
LR_INIT      = 3e-4
N_EPOCHS     = 10
N_STEPS      = 2048
BATCH_SIZE   = 256
GAMMA        = 0.995
GAE_LAMBDA   = 0.95
CLIP_RANGE   = 0.2
N_ENVS       = 4    # was 8 — lower contention on mp.Manager / PPO.load races

# === Self-play / checkpoints ===
TOTAL_STEPS         = 64_000_000   # ceiling — time-limit hits first
CHECKPOINT_EVERY    = 2_000_000
# Snapshot frequency — every 2M virtual steps (was 1M). Less I/O contention =
# less chance of save/load race conditions taking out a worker.
FROZEN_OPP_EVERY    = 2_000_000
# Cap on cached self-play models per worker. Each cached model uses ~1MB of
# RAM, plus inference allocations. With 8 workers, 8 cached = ~64MB total
# which is comfortably within Kaggle's RAM. Was 12 — lowered for headroom.
SELF_POOL_MAX       = 8

# === Bilateral training (Path B) ===
BILATERAL_TRAINING  = True         # randomize agent_team per episode

# === CQB RUSH REWARD STRUCTURE ===
# Goal: aggressive close-quarters specialist. Rewards trades and rush-in
# kills — 1-for-1 trade nets +20 (strongly encourages aggression),
# low survival pressure means the policy doesn't pull back, low team-lead
# means it doesn't wait for backup. Pair with SMG / SHOTGUN at deploy.
COEF_DMG_DEALT      = 0.5          # damage dealt valued slightly more
COEF_DMG_TAKEN      = 0.4          # taking damage hurts less than dealing
COEF_KILL           = 60.0         # very high — rush kills are the play
COEF_DEATH          = 60.0         # 1-for-1 trade = +20 net (aggression rewarded)
COEF_ALIVE          = 0.01         # minimal survival pressure
COEF_TEAM_LEAD      = 0.005        # don't wait for the team — push solo
COEF_EPISODE_WIN    = 50.0

# === TACTICAL reward shaping ===
# These activate the new combat_env.py reward terms. Tune with care:
# too aggressive on coef_suppress and the agent starts spray-and-pray
# at empty walls. The defaults below land in 'fires when it expects
# a hit OR when an ally is pushing' territory.
COEF_SUPPRESS        = 0.05    # per bullet within 60u of any enemy
COEF_SAFE_RELOAD     = 0.04    # per tick reloading while LoS-blocked
COEF_EXPOSED_RELOAD  = 0.08    # per tick reloading while visible to enemy
COEF_COVER_FIRE      = 0.06    # per fire-tick while ally is advancing

# Lock curriculum to stage 4 (deployment scale, no aim-cone shaping)
FORCE_DEPLOY_STAGE   = True
         # match win still counts but smaller
DISABLE_RESPAWN     = False        # match the deployed game

import os
OUTPUT_DIR = '/kaggle/working/ppo_ckpt'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=== TACTICAL training ===')
print(f'TIME_LIMIT_HOURS = {TIME_LIMIT_HOURS}h')
print(f'COEF_SUPPRESS={COEF_SUPPRESS} COEF_SAFE_RELOAD={COEF_SAFE_RELOAD} '
      f'COEF_EXPOSED_RELOAD={COEF_EXPOSED_RELOAD} COEF_COVER_FIRE={COEF_COVER_FIRE}')
print(f'BILATERAL_TRAINING = {BILATERAL_TRAINING}')
print(f'ENT_COEF: {ENT_COEF_INIT_CONT} -> {ENT_COEF_FINAL_CONT}')


## 2. Install

In [ ]:
%pip install -q stable-baselines3 onnx onnxruntime

## 3. Materialize combat_env.py

In [ ]:
# Write combat_env.py from a base64 blob — keeps the notebook
# self-contained for Kaggle uploads. Drops it directly into the
# working dir so `import combat_env` picks it up at module level
# (matches what cells 10+ expect).
import os, sys, base64
COMBAT_ENV_B64 = '''...'''
_module_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
_env_path = os.path.join(_module_dir, 'combat_env.py')
with open(_env_path, 'wb') as f:
    f.write(base64.b64decode(COMBAT_ENV_B64))
if _module_dir not in sys.path:
    sys.path.insert(0, _module_dir)
# Mirror CQB import pattern: BOTH module + named symbols
import combat_env
from combat_env import (
    CombatEnv, SinglePerspectiveEnv, Curriculum, ACTION_DIM, OBS_DIM,
    DEPLOY_WORLD_W, DEPLOY_WORLD_H, TICK_RATE, SQUAD_SIZE, DEFAULT_MATCH_TICKS,
    NN_MAG_SIZE, NN_RELOAD_FRAMES,
    random_opponent, idle_opponent, runner_opponent, make_ga_opponent,
    pick_map, curriculum_for_step,
)
# Also need make_curriculum_opponent_picker for cell 10
try:
    from combat_env import make_curriculum_opponent_picker
except ImportError:
    pass
print(f'combat_env loaded — OBS_DIM={OBS_DIM}, mag={NN_MAG_SIZE}, reload={NN_RELOAD_FRAMES}f')

# === Tactical shaping forward-port ===
# Patch curriculum_for_step so every stage-4 episode also carries the
# new tactical reward coefficients.
import combat_env as _cenv
_orig_cur_for_step = _cenv.curriculum_for_step
def _tactical_cur_for_step(step, total_steps):
    cur = _orig_cur_for_step(step, total_steps)
    cur.coef_dmg_dealt    = COEF_DMG_DEALT
    cur.coef_dmg_taken    = COEF_DMG_TAKEN
    cur.coef_kill         = COEF_KILL
    cur.coef_death        = COEF_DEATH
    cur.coef_alive        = COEF_ALIVE
    cur.coef_team_lead    = COEF_TEAM_LEAD
    cur.coef_episode_win  = COEF_EPISODE_WIN
    cur.coef_suppress       = COEF_SUPPRESS
    cur.coef_safe_reload    = COEF_SAFE_RELOAD
    cur.coef_exposed_reload = COEF_EXPOSED_RELOAD
    cur.coef_cover_fire     = COEF_COVER_FIRE
    if FORCE_DEPLOY_STAGE:
        cur.world_w = DEPLOY_WORLD_W
        cur.world_h = DEPLOY_WORLD_H
        cur.spawn_dist = 700.0
        cur.coef_visibility = 0.0
        cur.coef_approach   = 0.0
        cur.coef_aimcone    = 0.0
    return cur
_cenv.curriculum_for_step = _tactical_cur_for_step
print('tactical curriculum patch installed')


## 4. Find existing checkpoints + self-snapshots

Searches `/kaggle/working/` and `/kaggle/input/**/` for any
`ppo_combat_*.zip` AND `self_snap_*.zip`. Re-zips Kaggle's auto-extracted
directories the same way `finalize_ppo.ipynb` does.

In [ ]:
import re, glob, shutil

def _find_zips(name_pattern):
    """Find both raw .zip and Kaggle's auto-extracted directories matching pattern.
    Returns list of (step_number, path_to_zip)."""
    found = []
    seen = set()
    roots = ['/kaggle/working/ppo_ckpt', '/kaggle/working', '/kaggle/input']
    repacked = '/kaggle/working/repacked_ckpts'
    os.makedirs(repacked, exist_ok=True)

    # Phase 1: raw .zip files
    for root in roots:
        if not os.path.exists(root):
            continue
        for path in glob.glob(os.path.join(root, '**/*.zip'), recursive=True):
            if path in seen:
                continue
            name = os.path.basename(path)
            m = re.match(name_pattern, name)
            if m:
                step = int(m.group(1))
                found.append((step, path))
                seen.add(path)

    # Phase 2: extracted directories
    base_pattern = name_pattern.replace(r'\.zip$', '').replace(r'(\d+)', r'(\d+)')
    dir_re = re.compile(base_pattern.rstrip('$'))
    is_ppo_search = 'ppo_combat' in name_pattern
    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if 'policy.pth' in filenames and 'data' in filenames:
                base = os.path.basename(dirpath)
                m = dir_re.match(base) or re.search(r'(\d+)', base)
                if m:
                    step = int(m.group(1))
                else:
                    # No step number in dirname (e.g. dataset named 'dataaaa').
                    # Only accept as ppo_combat (never self_snap), use a huge
                    # sentinel step so it sorts as the latest checkpoint.
                    if not is_ppo_search:
                        continue
                    step = 99_000_000 + len(found)
                kind_prefix = 'ppo_combat' if is_ppo_search else 'self_snap'
                zip_path = os.path.join(repacked, f'{kind_prefix}_{step}.zip')
                if not os.path.exists(zip_path):
                    print(f'  Re-zipping {kind_prefix} step {step:,}: {dirpath}')
                    shutil.make_archive(zip_path[:-4], 'zip', dirpath)
                if zip_path not in seen:
                    found.append((step, zip_path))
                    seen.add(zip_path)
    found.sort(key=lambda x: x[0])
    return found

ckpts     = _find_zips(r'ppo_combat_(\d+)\.zip$')
selfsnaps = _find_zips(r'self_snap_(\d+)\.zip$')

print(f'\nFound {len(ckpts)} ppo_combat checkpoints:')
for s, p in ckpts:
    print(f'  step {s:>12,}  {p}')

print(f'\nFound {len(selfsnaps)} self_snap snapshots:')
for s, p in selfsnaps[:6]:
    print(f'  step {s:>12,}  {p}')
if len(selfsnaps) > 6: print(f'  ... and {len(selfsnaps)-6} more')

if not ckpts:
    print('\n!!! Diagnostic of /kaggle/input/:')
    if os.path.exists('/kaggle/input'):
        for d in sorted(os.listdir('/kaggle/input')):
            full = f'/kaggle/input/{d}'
            print(f'  {full}/')
            try:
                for item in sorted(os.listdir(full))[:10]:
                    print(f'    {item}')
            except OSError as e:
                print(f'    error: {e}')
    raise RuntimeError('No ppo_combat checkpoint found anywhere — see diagnostic above')

# Crash-recovery: if a previous run wrote ppo_combat_LATEST.zip in OUTPUT_DIR,
# prefer it over the original input checkpoint so we resume from the most
# recent training state instead of restarting from scratch.
latest_path = os.path.join(OUTPUT_DIR, 'ppo_combat_LATEST.zip')
if os.path.exists(latest_path) and os.path.getsize(latest_path) > 1024:
    resume_path = latest_path
    resume_step = ckpts[-1][0]    # we don't know the real step, just use last known
    print(f'\n>>> Found LATEST checkpoint from previous (crashed?) run — resuming from {resume_path}')
else:
    resume_step, resume_path = ckpts[-1]
    print(f'\n>>> Resuming from {resume_path} (step {resume_step:,})')

## 5. Build vec env with bilateral agent_team + pre-filled self pool

Differs from the original training notebook in three ways:
  - agent_team_provider returns random 0/1 (was always 0)
  - opponent_factory uses GA-best when agent is team 0 (since existing
    snapshots are team-0-trained), and snapshots when agent is team 1
  - self pool is pre-filled with existing self_snap_* + the last few
    ppo_combat checkpoints

In [ ]:
import multiprocessing as mp
import numpy as np
import random
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor

# Fallback defaults — picked up from CONFIG cell if defined, otherwise use these.
# Lets the cell run standalone after the user upgrades from an older CONFIG.
COEF_DMG_DEALT     = globals().get('COEF_DMG_DEALT',     0.5)
COEF_DMG_TAKEN     = globals().get('COEF_DMG_TAKEN',     0.4)
COEF_KILL          = globals().get('COEF_KILL',          50.0)
COEF_DEATH         = globals().get('COEF_DEATH',         40.0)
COEF_ALIVE         = globals().get('COEF_ALIVE',         0.01)
COEF_TEAM_LEAD     = globals().get('COEF_TEAM_LEAD',     0.005)
COEF_EPISODE_WIN   = globals().get('COEF_EPISODE_WIN',   100.0)
SHAPING_DECAY_FRAC = globals().get('SHAPING_DECAY_FRAC', 0.10)
BILATERAL_TRAINING = globals().get('BILATERAL_TRAINING', True)
SELF_POOL_MAX      = globals().get('SELF_POOL_MAX',      12)

DEFAULT_GA_GENOME = {
    'view_arc': 2.4, 'view_range': 720, 'snap_to_threat': 0.27,
    'patrol_scan_speed': 0.04, 'engage_distance': 280, 'spread': 0.07,
    'fire_cd_frames': 12, 'cover_dmg_threshold': 28, 'peek_duration': 55,
    'hide_duration': 40, 'flee_hp_pct': 0.34, 'flank_chance': 0.6,
}

# Shared state across workers
_mp_manager = mp.Manager()
_global_step      = _mp_manager.Value('i', 0)
_self_pool_paths  = _mp_manager.list()

# Pre-fill the pool ONLY with self_snap_*.zip (real frozen snapshots from
# previous training runs). Don't pre-fill with the resume checkpoint itself
# (it's the SAME zip we just resumed from, so 8 workers would all race to
# load it on first reset → deadlock). New self-snap snapshots get added by
# the callback as training progresses.
PREFILL_SELF_POOL = True   # set False if you hit deadlocks on first reset
if PREFILL_SELF_POOL and selfsnaps:
    for _, p in selfsnaps[-SELF_POOL_MAX:]:
        _self_pool_paths.append(p)
    print(f'Pre-filled self-play pool with {len(_self_pool_paths)} self_snap models')
else:
    print(f'Self-play pool starts empty (will fill as training generates snapshots)')

_worker_policy_cache = {}

def _load_policy(path):
    # Load (and cache) a self-play snapshot. Hardened: sanity-check before
    # caching; runtime predict wrapped in try/except so a bad call never
    # propagates -> kills worker -> kills training.
    if path in _worker_policy_cache:
        return _worker_policy_cache[path]
    if not os.path.exists(path):
        return None
    # Safety: reject empty or partially-written files
    try:
        if os.path.getsize(path) < 1024:
            return None
    except OSError:
        return None
    try:
        from stable_baselines3 import PPO as _PPO
        m = _PPO.load(path, device='cpu')
        # Smoke-check: catch corrupt models BEFORE caching them
        _test_obs = np.zeros((1, OBS_DIM), dtype=np.float32)
        m.predict(_test_obs, deterministic=True)
    except Exception as e:
        print(f'  [worker] load failed for {path}: {type(e).__name__}: {e}')
        return None

    def _policy(obs_list, env, _m=m, _p=path):
        try:
            obs_arr = np.stack(obs_list).astype(np.float32)
            actions, _ = _m.predict(obs_arr, deterministic=False)
            return [int(a) for a in actions]
        except Exception as e:
            # If the model misbehaves at runtime, evict it from cache and
            # fall back to random actions instead of crashing the worker.
            print(f'  [worker] predict failed for {_p}: {type(e).__name__}: {e}')
            _worker_policy_cache.pop(_p, None)
            return [random.randint(0, ACTION_DIM - 1) for _ in obs_list]
    _worker_policy_cache[path] = _policy
    return _policy

def _safe_save(model_obj, path):
    # Atomic save: write to .tmp then rename. Prevents workers from reading
    # a partially-written checkpoint and segfaulting on it.
    tmp = path + '.tmp'
    model_obj.save(tmp)
    os.replace(tmp, path)   # atomic rename on Linux/macOS

# === Curriculum: locked to stage 4 with optional decaying shaping early on ===
def continue_curriculum_for_step(step, total):
    p = max(0.0, min(1.0, step / max(1, total)))
    s = max(0.0, 1.0 - p / max(SHAPING_DECAY_FRAC, 1e-6))   # 1 → 0 over first SHAPING_DECAY_FRAC
    return Curriculum(
        world_w=DEPLOY_WORLD_W,
        world_h=DEPLOY_WORLD_H,
        spawn_dist=700.0,
        match_ticks=DEFAULT_MATCH_TICKS,
        p_static=0.0, p_runner=0.0,
        p_random=0.05,
        p_ga=0.40,
        p_self=0.55,
        # Aggressive-smart reward structure (override the defaults)
        coef_dmg_dealt=COEF_DMG_DEALT,
        coef_dmg_taken=COEF_DMG_TAKEN,
        coef_kill=COEF_KILL,
        coef_death=COEF_DEATH,
        coef_alive=COEF_ALIVE,
        coef_team_lead=COEF_TEAM_LEAD,
        coef_episode_win=COEF_EPISODE_WIN,
        # Light shaping early to ease the bilateral transition
        coef_visibility=0.02 * s,
        coef_approach=0.001 * s,
        coef_aimcone=0.004 * s,
    )

def make_opponent_for_curriculum(c):
    pool = []
    if c.p_self > 0 and len(_self_pool_paths) > 0:
        for p in list(_self_pool_paths)[-SELF_POOL_MAX:]:
            if not os.path.exists(p):
                continue
            pol = _load_policy(p)
            if pol is not None:
                pool.append(pol)
    picker = make_curriculum_opponent_picker(
        ga_genome=DEFAULT_GA_GENOME, self_pool=pool)
    return picker(c)

def make_env(rank: int):
    def _init():
        seed = 2000 + rank * 9973
        rng = random.Random(seed)
        def cur_provider():
            return continue_curriculum_for_step(_global_step.value, TOTAL_STEPS)
        def team_provider():
            return rng.randint(0, 1) if BILATERAL_TRAINING else 0
        return SinglePerspectiveEnv(
            curriculum_provider=cur_provider,
            opponent_factory=make_opponent_for_curriculum,
            agent_team_provider=team_provider,
            squad_size=SQUAD_SIZE, seed=seed,
        )
    return _init

# Quick single-env smoke
test_env = make_env(0)()
obs, _ = test_env.reset(seed=42)
total = 0
team_counts = {0: 0, 1: 0}
for _ in range(50):
    a = test_env.action_space.sample()
    o, r, d, t, info = test_env.step(a)
    total += r
    if d:
        team_counts[test_env._agent_team] = team_counts.get(test_env._agent_team, 0) + 1
        test_env.reset()
print(f'Single env smoke: 50-step reward={total:.2f}, team counts after resets: {team_counts}')

print(f'Building SubprocVecEnv with {N_ENVS} workers (start_method=fork)...')
vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)], start_method='fork')
vec_env = VecMonitor(vec_env)
print('Vec env ready.')

## 6. Load existing PPO model + attach to vec env

In [ ]:
import torch
from stable_baselines3 import PPO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading {resume_path} on {device} ...')
# IMPORTANT: load fresh on the target device (don't reuse a CPU-loaded
# model from the ONNX-export cell — that ran 'device=cpu' which would
# leave self.device='cpu' and training would NOT use the GPU).
model = PPO.load(resume_path, env=vec_env, device=device,
                 custom_objects={'learning_rate': LR_INIT,
                                 'lr_schedule':   lambda _: LR_INIT,
                                 'clip_range':    lambda _: CLIP_RANGE})
# Override knobs that shouldn't be inherited from the previous training run
model.ent_coef    = ENT_COEF_INIT_CONT
model.n_epochs    = N_EPOCHS
model.batch_size  = BATCH_SIZE
model.gamma       = GAMMA
model.gae_lambda  = GAE_LAMBDA
# Confirm GPU usage
_param_dev = next(model.policy.parameters()).device
print(f'  Loaded. policy on device: {_param_dev}, params={sum(p.numel() for p in model.policy.parameters()):,}')
print(f'  ent_coef={model.ent_coef}, n_epochs={model.n_epochs}, gamma={model.gamma}')
if str(_param_dev) == 'cpu' and torch.cuda.is_available():
    print('  WARN: model is on CPU but GPU is available — performance will be poor!')

## 7. Training callback (time-budget + curriculum + self-play)

In [ ]:
import time
from stable_baselines3.common.callbacks import BaseCallback

class ContinueCallback(BaseCallback):
    def __init__(self, output_dir, deadline_ts, verbose=0):
        super().__init__(verbose)
        self.output_dir = output_dir
        self.deadline_ts = deadline_ts
        self.t_start = time.time()
        self.last_freeze = 0
        self.last_checkpoint = 0
        self.last_log_t = 0.0
        self.stopped_for_time = False

    def _on_step(self):
        n = self.num_timesteps
        elapsed = time.time() - self.t_start
        deadline_elapsed = self.deadline_ts - self.t_start
        time_progress = max(0.0, min(1.0, elapsed / max(1.0, deadline_elapsed)))
        virtual_step = int(time_progress * TOTAL_STEPS)
        _global_step.value = virtual_step

        # Linear ent_coef decay
        self.model.ent_coef = ENT_COEF_FINAL_CONT + (1 - time_progress) * (ENT_COEF_INIT_CONT - ENT_COEF_FINAL_CONT)

        if time.time() >= self.deadline_ts:
            print(f'\n[time] deadline at step {n:,} ({elapsed/3600:.2f}h elapsed) — stopping.')
            self.stopped_for_time = True
            return False

        # Self-play snapshot — atomic save to avoid race with workers.
        # Skipped in SMOKE_TEST mode: smoke compresses 64M virtual steps into
        # 180 wall-seconds, so FROZEN_OPP_EVERY=1M would fire every ~3s and
        # waste tens of wall-seconds on zip-save IO during the 3-min check.
        if not SMOKE_TEST and virtual_step - self.last_freeze >= FROZEN_OPP_EVERY:
            self.last_freeze = virtual_step
            path = os.path.join(self.output_dir, f'self_snap_{n}.zip')
            try:
                _safe_save(self.model, path)
                _self_pool_paths.append(path)
                while len(_self_pool_paths) > SELF_POOL_MAX:
                    old = _self_pool_paths.pop(0)
                    try:
                        # Only delete files we created in OUTPUT_DIR — never
                        # touch user-uploaded inputs or the repacked dir.
                        if old.startswith(self.output_dir) and 'self_snap' in os.path.basename(old):
                            if os.path.exists(old):
                                os.remove(old)
                    except OSError: pass
                if self.verbose:
                    print(f'  [selfplay] snap @ step {n:,} pool={len(_self_pool_paths)}')
            except Exception as e:
                print(f'  [selfplay] snap failed: {e}')

        # Checkpoint — atomic save (also skipped in smoke mode; first call
        # fires immediately at n>>CHECKPOINT_EVERY since num_timesteps=18M+)
        if not SMOKE_TEST and n - self.last_checkpoint >= CHECKPOINT_EVERY:
            self.last_checkpoint = n
            path = os.path.join(self.output_dir, f'ppo_combat_cqb_{n}.zip')
            try:
                _safe_save(self.model, path)
                if self.verbose:
                    print(f'  [ckpt] @ step {n:,} -> {path}')
            except Exception as e:
                print(f'  [ckpt] save failed: {e}')

        # Always keep a "latest" pointer for crash recovery — atomic save every
        # 200k real steps. If training dies, this is what you reload from.
        if n - getattr(self, '_last_latest', 0) >= 200_000:
            self._last_latest = n
            try:
                _safe_save(self.model, os.path.join(self.output_dir, 'ppo_combat_LATEST.zip'))
            except Exception as e:
                print(f'  [latest] save failed: {e}')

        # Status log every 2 minutes
        if elapsed - self.last_log_t >= 120:
            self.last_log_t = elapsed
            sps = n / max(1.0, elapsed)
            eta_h = (deadline_elapsed - elapsed) / 3600
            print(f'[t+{elapsed/60:>6.1f}m] step {n:>10,} ({sps:>5.0f}/s, eta {eta_h:>4.2f}h) '
                  f'ent={self.model.ent_coef:.4f} pool={len(_self_pool_paths)}')
        return True

print('Callback defined.')

## 8. Train (long-running)

In [ ]:
# Two-belt SB3 silence — long Kaggle runs accumulate output as DOM nodes
# in the browser tab; SB3's per-rollout table is the worst offender. Belt 1
# replaces the model's logger with a no-output silent one, belt 2 sets
# log_interval so high it never naturally triggers in real-mode training.
from stable_baselines3.common.logger import Logger
model._logger = Logger(folder=None, output_formats=[])
model.verbose = 0
print('SB3 internal logging silenced.')

deadline_ts = time.time() + TIME_LIMIT_HOURS * 3600
print(f'>>> Continuing training until {time.strftime("%H:%M:%S", time.localtime(deadline_ts))} '
      f'({TIME_LIMIT_HOURS}h)')

t0 = time.time()
cb = ContinueCallback(output_dir=OUTPUT_DIR, deadline_ts=deadline_ts, verbose=1)
# Smoke test wants frequent logs (so you see progress in 3 min); real mode
# pushes log_interval high so SB3 essentially never dumps even if the
# silenced logger above missed something.
LOG_INTERVAL = 2 if SMOKE_TEST else 1000
model.learn(total_timesteps=TOTAL_STEPS, callback=cb,
            progress_bar=False, reset_num_timesteps=False,
            log_interval=LOG_INTERVAL)

elapsed = time.time() - t0
print(f'\n>>> Done. Wall: {elapsed/3600:.2f}h, total steps: {model.num_timesteps:,}')

final_path = os.path.join(OUTPUT_DIR, 'ppo_combat_tactical_final.zip')
model.save(final_path)
print(f'>>> Final saved: {final_path}')

## 9. Export ONNX

In [ ]:
import torch.nn as nn
import os, glob, re, shutil
from stable_baselines3 import PPO

# Auto-load if model isn't in scope (kernel restart, skipped training cell, etc.)
# Handles three discovery cases:
#   (1) raw .zip in working dir,
#   (2) raw .zip in /kaggle/input,
#   (3) Kaggle auto-extracted SB3 model directory (single-zip dataset upload
#       gets expanded into a dir containing policy.pth + data + ...) — we
#       re-zip these into /kaggle/working/repacked_ckpts/ so PPO.load works.
if 'model' not in dir() or model is None:
    candidates = []
    for path in glob.glob(os.path.join(OUTPUT_DIR, '*.zip')):
        name = os.path.basename(path)
        m = re.match(r'ppo_combat_cqb_(\d+)\.zip$', name) \
            or re.match(r'ppo_combat_(\d+)\.zip$', name)
        if m:
            prefix = 0 if 'continued' in name else 1
            candidates.append((prefix, int(m.group(1)), path))
    for path in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        m = re.search(r'(\d+)', os.path.basename(path))
        if m: candidates.append((2, int(m.group(1)), path))
    repacked = '/kaggle/working/repacked_ckpts'
    os.makedirs(repacked, exist_ok=True)
    for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
        if 'policy.pth' in filenames and 'data' in filenames:
            base = os.path.basename(dirpath)
            m = re.search(r'(\d+)', base)
            step = int(m.group(1)) if m else 0
            zip_path = os.path.join(repacked, f'recovered_{step or "x"}.zip')
            if not os.path.exists(zip_path):
                print(f'Re-zipping {dirpath} -> {zip_path}')
                shutil.make_archive(zip_path[:-4], 'zip', dirpath)
            candidates.append((3, step, zip_path))

    if not candidates:
        print('!!! Nothing found. Diagnostic of /kaggle/input/:')
        if os.path.exists('/kaggle/input'):
            for d in sorted(os.listdir('/kaggle/input')):
                full = f'/kaggle/input/{d}'
                print(f'  {full}/')
                try:
                    for item in sorted(os.listdir(full))[:8]:
                        print(f'    {item}')
                except OSError as e:
                    print(f'    (error: {e})')
        raise RuntimeError(f'No model in scope and no checkpoint found anywhere')
    candidates.sort()
    chosen = candidates[0][2]
    print(f'Auto-loading {chosen}')
    model = PPO.load(chosen, device='cpu')


class OnnxablePolicy(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net
    def forward(self, obs):
        latent_pi = self.extractor.policy_net(obs)
        logits = self.action_net(latent_pi)
        return torch.softmax(logits, dim=-1)

model.policy.to('cpu')
onnxable = OnnxablePolicy(model.policy).eval()
dummy = torch.zeros(1, OBS_DIM, dtype=torch.float32)
onnx_path = os.path.join(OUTPUT_DIR, 'model_tactical.onnx')
torch.onnx.export(
    onnxable, dummy, onnx_path,
    input_names=['obs'], output_names=['action_probs'],
    dynamic_axes={'obs': {0: 'batch'}, 'action_probs': {0: 'batch'}},
    opset_version=17, dynamo=False,
)
print(f'ONNX -> {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
test_in = np.random.randn(1, OBS_DIM).astype(np.float32)
out = sess.run(None, {'obs': test_in})
print(f'  inference: {test_in.shape} -> {out[0].shape}, sum={out[0][0].sum():.3f}')

## 10. Sanity check: NN vs GA-best from BOTH spawn sides

The whole point of bilateral training is that the model is now competent
on team 0 AND team 1. Run 10 matches as each, report combined win rate.

In [ ]:
# Self-contained: inline what this cell needs (genome + curriculum) so it
# doesn't depend on training-cell scope when running standalone.
import numpy as np
from combat_env import CombatEnv, Curriculum, make_ga_opponent, SQUAD_SIZE
import combat_env as _ce

DEFAULT_GA_GENOME = DEFAULT_GA_GENOME if 'DEFAULT_GA_GENOME' in dir() else {
    'view_arc': 2.4, 'view_range': 720, 'snap_to_threat': 0.27,
    'patrol_scan_speed': 0.04, 'engage_distance': 280, 'spread': 0.07,
    'fire_cd_frames': 12, 'cover_dmg_threshold': 28, 'peek_duration': 55,
    'hide_duration': 40, 'flee_hp_pct': 0.34, 'flank_chance': 0.6,
}
if 'model' not in dir() or model is None:
    raise RuntimeError('No model loaded — run the ONNX export cell first to auto-load it.')

ga_policy = make_ga_opponent(DEFAULT_GA_GENOME)
# Deployment-scale, no shaping — clean judgment
deploy_c = Curriculum(
    world_w=_ce.DEPLOY_WORLD_W, world_h=_ce.DEPLOY_WORLD_H,
    spawn_dist=700.0, match_ticks=_ce.DEFAULT_MATCH_TICKS,
    coef_visibility=0.0, coef_approach=0.0, coef_aimcone=0.0,
)

print('Running 10 matches as team 0 + 10 as team 1 (deployment scale)...')

results_per_team = {0: {'W':0, 'L':0, 'D':0}, 1: {'W':0, 'L':0, 'D':0}}
for agent_team in (0, 1):
    for match in range(10):
        env = CombatEnv(opponent_policy=ga_policy, curriculum=deploy_c, seed=30000 + match * 11)
        env.reset(seed=30000 + match * 11)
        obs_list = env._observe_team(team=agent_team)
        for _ in range(env.match_ticks):
            actions = []
            for k in range(SQUAD_SIZE):
                obs = obs_list[k].astype(np.float32)
                a, _ = model.predict(obs, deterministic=True)
                actions.append(int(a))
            obs_list, rewards, done, info = env.step(actions, agent_team=agent_team)
            if done: break
        ka = env.team_kills[agent_team]
        ko = env.team_kills[1 - agent_team]
        if ka > ko:    results_per_team[agent_team]['W'] += 1
        elif ko > ka:  results_per_team[agent_team]['L'] += 1
        else:          results_per_team[agent_team]['D'] += 1

print(f'\nAs team 0 (blue/left):  W/L/D = {results_per_team[0]}')
print(f'As team 1 (red/right):  W/L/D = {results_per_team[1]}')
total_W = results_per_team[0]['W'] + results_per_team[1]['W']
print(f'\nCombined: {total_W}/20 wins ({total_W*5}%)')
if total_W >= 14: print('[EXCELLENT] Strong bilateral player.')
elif total_W >= 10: print('[OK] Comparable to GA-best on both sides.')
else: print('[WEAK] Try more steps or check obs distributions.')

## 11. Output

- `/kaggle/working/ppo_ckpt/model.onnx` ← drop into `ai_arena/onnx/model.onnx`
  (you can also copy it to `ai_arena/onnx/model_hard.onnx` since this IS your
  new strongest model)
- `ppo_combat_continued_<N>.zip` for further runs

After dropping the new model.onnx, you can REMOVE the JS-side mirroring
code (`flipX`, `NN_ACTION_MIRROR_X`) since the model now handles team 1
natively — but it doesn't hurt to leave it (the mirror just becomes a
no-op for a bilateral model).